# Desaparecidos na Cidade de São Paulo (2015–2025)
**Reportagem:** [Vidas em suspenso: Periferias registram maior número de desaparecidos em SP](https://agenciamural.org.br/especiais/desaparecidos-em-sp/) — *Agência Mural*

Este notebook contém o pipeline completo de extração, limpeza, deduplicação e análise exploratória dos Boletins de Ocorrência (BOs) de desaparecimento de pessoas registrados na cidade de São Paulo entre **janeiro de 2015 e julho de 2025**, obtidos via Lei de Acesso à Informação (LAI / FalaSP).

In [4]:
!pip install pyxlsb openpyxl -q

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("husl")

os.makedirs('outputs', exist_ok=True)

PATH_2015_2019 = 'Dados/FalaSP 2025072815313670 - RDO 2015 a 2019.xlsb'
PATH_2020_2022 = 'Dados/FalaSP 2025072815313670 - RDO 2020 a 2022 e Metodologia.xlsb'
PATH_2023_2025 = 'Dados/FalaSP 2025072815313670 - SPJ 2023 a 2025.xlsb'

print("Etapa 1: Carregando planilhas")
df_2015_2019 = pd.read_excel(PATH_2015_2019, engine='pyxlsb')
df_2020_2022 = pd.read_excel(PATH_2020_2022, engine='pyxlsb')
df_2023_2025 = pd.read_excel(PATH_2023_2025, engine='pyxlsb')

total_bruto = len(df_2015_2019) + len(df_2020_2022) + len(df_2023_2025)
print(f" 2015–2019 (RDO): {len(df_2015_2019):,} registros ({df_2015_2019.shape[1]} colunas)")
print(f" 2020–2022 (RDO): {len(df_2020_2022):,} registros ({df_2020_2022.shape[1]} colunas)")
print(f" 2023–2025 (SPJ): {len(df_2023_2025):,} registros ({df_2023_2025.shape[1]} colunas)")
print(f" Total bruto carregado: {total_bruto:,} registros")

Etapa 1: Carregando planilhas
 2015–2019 (RDO): 127,433 registros (57 colunas)
 2020–2022 (RDO): 58,418 registros (57 colunas)
 2023–2025 (SPJ): 48,662 registros (60 colunas)
 Total bruto carregado: 234,513 registros


In [7]:
print("Etapa 2 e 3: Padronizar colunas e unificando bases")

# Remover colunas duplicadas internas nos arquivos originais
df_2015_2019 = df_2015_2019.loc[:, ~df_2015_2019.columns.duplicated(keep='first')].copy()
df_2020_2022 = df_2020_2022.loc[:, ~df_2020_2022.columns.duplicated(keep='first')].copy()
df_2023_2025 = df_2023_2025.loc[:, ~df_2023_2025.columns.duplicated(keep='first')].copy()

# Padronizar nomenclatura da base SPJ (2023-2025) para o padrão RDO
renomear_spj = {'NOME_MUNICIPIO': 'CIDADE', 'COR_CURTIS': 'COR_CUTIS'}
df_2023_2025 = df_2023_2025.rename(columns={k: v for k, v in renomear_spj.items() if k in df_2023_2025.columns})
df_2023_2025 = df_2023_2025.loc[:, ~df_2023_2025.columns.duplicated(keep='first')].copy()

# Interseção das 51 colunas comuns às três bases
colunas_comuns = sorted(list(
    set(df_2015_2019.columns) &
    set(df_2020_2022.columns) &
    set(df_2023_2025.columns)
))

df_completo = pd.concat([
    df_2015_2019[colunas_comuns],
    df_2020_2022[colunas_comuns],
    df_2023_2025[colunas_comuns]
], ignore_index=True)

print(f"Colunas comuns identificadas: {len(colunas_comuns)}")
print(f"Base unificada (Estado SP): {len(df_completo):,} registros")

# Etapa 4: Remoção de duplicatas conforme critério oficial da SSP-SP
print("Etapa 4: Removendo duplicatas (NOME_DELEGACIA + ANO_BO + NUM_BO)...")
registros_antes = len(df_completo)
df_completo = df_completo.drop_duplicates(
    subset=['NOME_DELEGACIA', 'ANO_BO', 'NUM_BO'],
    keep='first'
).copy()
registros_depois = len(df_completo)

print(f"Duplicatas removidas: {registros_antes - registros_depois:,}")
print(f"Registros únicos no Estado de SP: {registros_depois:,}")

Etapa 2 e 3: Padronizar colunas e unificando bases
Colunas comuns identificadas: 51
Base unificada (Estado SP): 234,513 registros
Etapa 4: Removendo duplicatas (NOME_DELEGACIA + ANO_BO + NUM_BO)...
Duplicatas removidas: 9,735
Registros únicos no Estado de SP: 224,778


In [9]:
print("Etapa 5 e 6: Filtrando município de São Paulo e tratando variáveis...")

# Filtro geográfico: capital aparece padronizada como 'S.PAULO'
df_sp = df_completo[df_completo['CIDADE'].astype(str).str.upper().str.strip() == 'S.PAULO'].copy()
print(f"Total de BOs únicos na cidade de São Paulo: {len(df_sp):,}")

# Referência temporal: uso de ANO_BO (conforme nota metodológica sobre datas XLSB)
df_sp['ANO_OCORRENCIA'] = df_sp['ANO_BO'].astype(int)

# Padronização de Cor/Raça (remoção de espaços extras e unificação de categorias)
mapa_cor = {
    'BRANCA': 'Branca',
    'PARDA': 'Parda',
    'PRETA': 'Preta',
    'AMARELA': 'Amarela',
    'VERMELHA': 'Indígena',
    'IGNORADA': 'Não informada',
    'NÃO INFORMADA': 'Não informada',
    'OUTROS': 'Outros',
    'ALBINA': 'Outros'
}
df_sp['COR_CUTIS_LIMPA'] = (
    df_sp['COR_CUTIS']
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .replace(mapa_cor)
)

# Faixas etárias (filtrando valores inválidos como negativos ou > 120 anos)
df_sp['IDADE_VALIDA'] = df_sp['IDADE_PESSOA'].where(
    (df_sp['IDADE_PESSOA'] >= 0) & (df_sp['IDADE_PESSOA'] <= 120),
    np.nan
)
df_sp['FAIXA_ETARIA'] = pd.cut(
    df_sp['IDADE_VALIDA'],
    bins=[-1, 12, 18, 25, 35, 45, 60, 120],
    labels=['0-12 anos', '13-18 anos', '19-25 anos', '26-35 anos', '36-45 anos', '46-60 anos', '60+ anos']
)

# Diagnóstico de dados faltantes (Capital)
print("Percentual de dados faltantes (Capital - 74.026 BOs):")
for col in ['BAIRRO', 'COR_CUTIS', 'IDADE_PESSOA', 'SEXO_PESSOA', 'NOME_DELEGACIA_CIRC']:
    nulos = df_sp[col].isna().sum()
    print(f"  • {col}: {nulos:,} ({(nulos / len(df_sp)) * 100:.1f}%)")

Etapa 5 e 6: Filtrando município de São Paulo e tratando variáveis...
Total de BOs únicos na cidade de São Paulo: 74,026
Percentual de dados faltantes (Capital - 74.026 BOs):
  • BAIRRO: 1,941 (2.6%)
  • COR_CUTIS: 14,479 (19.6%)
  • IDADE_PESSOA: 856 (1.2%)
  • SEXO_PESSOA: 1 (0.0%)
  • NOME_DELEGACIA_CIRC: 0 (0.0%)


In [11]:
print("=" * 70)
print("ETAPA 7: RESULTADOS CONSOLIDADOS — CIDADE DE SÃO PAULO")
print("=" * 70)

# 1. Evolução Temporal
print("DISTRIBUIÇÃO POR ANO (ANO_BO):")
evolucao_anual = df_sp['ANO_OCORRENCIA'].value_counts().sort_index()
print(evolucao_anual)

# 2. Ranking de Bairros (mantendo grafia original conforme decisão metodológica)
bairros_ranking = df_sp['BAIRRO'].dropna().value_counts()
print(f"TOP 20 BAIRROS (de {len(bairros_ranking):,} valores únicos registrados):")
print(bairros_ranking.head(20))

top5 = bairros_ranking.head(5).sum()
print(f"Filtrando os 5 bairros líderes somam {top5:,} BOs ({(top5 / len(df_sp)) * 100:.1f}% do total da capital)")

# 3. Ranking de Delegacias de Circunscrição (local real da ocorrência)
delegacias_ranking = df_sp['NOME_DELEGACIA_CIRC'].dropna().value_counts()
print(f"TOP 15 DELEGACIAS DE CIRCUNSCRIÇÃO (NOME_DELEGACIA_CIRC):")
print(delegacias_ranking.head(15))

# 4. Perfil Demográfico
print("PERFIL DEMOGRÁFICO:")
print("Sexo:")
print(pd.DataFrame({
    'Total': df_sp['SEXO_PESSOA'].value_counts(),
    '%': (df_sp['SEXO_PESSOA'].value_counts(normalize=True) * 100).round(1)
}))

print("Cor/Raça (entre registros preenchidos):")
print(pd.DataFrame({
    'Total': df_sp['COR_CUTIS_LIMPA'].value_counts(),
    '%': (df_sp['COR_CUTIS_LIMPA'].value_counts(normalize=True) * 100).round(1)
}))

print("Faixa Etária x Sexo:")
print(pd.crosstab(df_sp['FAIXA_ETARIA'], df_sp['SEXO_PESSOA'], margins=True))

ETAPA 7: RESULTADOS CONSOLIDADOS — CIDADE DE SÃO PAULO
DISTRIBUIÇÃO POR ANO (ANO_BO):
ANO_OCORRENCIA
2014       1
2015    8454
2016    6918
2017    6759
2018    6628
2019    6969
2020    5456
2021    5386
2022    5895
2023    7883
2024    8879
2025    4798
Name: count, dtype: int64
TOP 20 BAIRROS (de 6,016 valores únicos registrados):
BAIRRO
GRAJAU               1946
BRASILANDIA          1538
CIDADE ADEMAR        1456
CIDADE TIRADENTES    1436
SAPOPEMBA            1381
ITAQUERA             1287
CAPAO REDONDO        1255
JARDIM ANGELA        1159
SANTANA              1119
JD SÃO LUIS          1103
CAMPO LIMPO          1099
CIDADE DUTRA         1019
ITAIM PAULISTA        976
TREMEMBE              957
SACOMA                945
PARELHEIROS           910
JABAQUARA             899
JARAGUA               855
GUAIANASES            839
LAJEADO               834
Name: count, dtype: int64
Filtrando os 5 bairros líderes somam 7,757 BOs (10.5% do total da capital)
TOP 15 DELEGACIAS DE CIRCUNSCRIÇÃO 